# Shared model integration

All active base models use the same direction-32 predictor tables, candidate identities, ordered reference labels, training exclusions and saved trip folds. Only the estimator and its training-only preprocessing differ.

These definitions do not train a model. OOF rows must cover every valid training candidate, including no-positive groups, and use the shared point-normalized score format. Student-test predictor labels remain blank until scoring after prediction.

In [ ]:
%run 06_model_helpers.ipynb

shared_feature_columns = json.loads(Path("outputs/direction_features/feature_columns.json").read_text())
training_exclusions = json.loads(Path("outputs/training_exclusions.json").read_text())["excluded_trip_ids"]


def load_shared_training():
    data = pd.read_parquet("outputs/direction_features/train.parquet")
    data = data[data["label"].notna() & ~data["trip_id"].isin(training_exclusions)].reset_index(drop=True)
    data["label"] = data["label"].astype("int8")
    folds = pd.read_csv(contract["fold_file"], dtype={"trip_id": str}).set_index("group_id")["fold"]
    data["fold"] = data["group_id"].map(folds)
    points = pd.read_csv("outputs/candidates/points.csv", dtype={"trip_id": str})
    points = points[~points["bad_gps"] & ~points["trip_id"].isin(training_exclusions)].copy()
    points["fold"] = points["group_id"].map(folds)
    assert data["fold"].notna().all() and data["group_id"].is_monotonic_increasing
    assert data.groupby("trip_id")["fold"].nunique().eq(1).all()
    assert data["trip_id"].nunique() == 78 and len(data) == points["candidate_count"].sum()
    return data, points


def load_shared_student_test():
    data = pd.read_parquet("outputs/direction_features/student_test.parquet")
    assert data["label"].isna().all()
    points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str})
    return data, points


def score_student_test(data, probabilities, points):
    valid_points = points[~points["bad_gps"]].copy()
    valid = data["group_id"].isin(valid_points["group_id"])
    scoring = data.loc[valid, ["group_id", "segment_id"]].copy()
    truth = scoring["group_id"].map(valid_points.set_index("group_id")["true_arc_id"])
    scoring["label"] = scoring["segment_id"].eq(truth).astype("int8")
    return score_points(scoring, np.asarray(probabilities)[valid.to_numpy()], valid_points)


def register_model(name, record):
    path = Path("models/registry.json")
    registry = json.loads(path.read_text())
    registry["models"][name] = record
    path.write_text(json.dumps(registry, indent=2) + "\n")